# 🎯 Actividad: "Entrena tu propio detector de objetos"

## ¿Qué vas a construir?

En grupos de 2-3 personas vais a crear **un detector de objetos personalizado** capaz de reconocer un objeto real de vuestro entorno (aula, taller, pasillo) en imágenes nuevas y en directo con la webcam.

Para ello recorreréis el pipeline completo de un proyecto de visión por computador:

```
RECOPILAR fotos → ETIQUETAR → EXPORTAR en formato YOLO → ENTRENAR → PROBAR
     (móvil)      (Roboflow)      (Roboflow)           (Colab)   (webcam)
```

> 📚 **Base teórica:** esta actividad sigue el flujo del artículo *"Detección de objetos con YOLO"* (Linux Magazine nº 311) que hemos trabajado en clase.

## 📅 Duración y entrega

- **3 sesiones** (1: recopilar y etiquetar · 2: entrenar · 3: probar, evaluar y presentar)
- **Entrega:** este cuaderno completado (celdas de código + respuestas de texto) como `detector_objeto_NOMBREGRUPO.ipynb`
- **Presentación final:** 5 minutos por grupo mostrando el detector funcionando

## ✅ Evaluación (10 puntos)

| Apartado | Puntos |
|---|---|
| Dataset recopilado siguiendo la checklist de buenas prácticas | 2.5 |
| Etiquetado correcto en Roboflow (clase única, cajas ajustadas) | 1.5 |
| Entrenamiento completado y explicación de parámetros (Pregunta 2) | 2 |
| Evaluación honesta con fotos nuevas (Pregunta 3) | 2 |
| Reflexión final (Pregunta 4) | 2 |


# 📸 Fase 1 — Recopilar el dataset (checklist de buenas prácticas)

## Elige el objeto

Cualquier objeto cotidiano **propio** (no personas): auriculares, un tornillo, la tarjeta del comedor, un ratón, una llave, una botella concreta... Un solo objeto = **una sola clase**.

## La checklist (imprimidla o tenla en el móvil)

Cada fotografía que hagáis debe sumar variedad en alguna de estas direcciones. **El objetivo: que el dataset se parezca al mundo real donde el detector va a trabajar.**

| # | Buena práctica | ¿Por qué? (lo que "piensa" el modelo) | ¿Hecho? |
|---|---|---|---|
| 1 | **Mínimo 60 fotos** (ideal 100+) | El modelo no memoriza: *generaliza* patrones. Con pocas fotos solo ve casos particulares | ☐ |
| 2 | **Mínimo 5 ángulos** distintos (frontal, lateral, 3/4, superior, desde abajo) | Si solo hay un ángulo, aprende "objeto = cosa vista así", no el objeto | ☐ |
| 3 | **Mínimo 3 iluminaciones** (natural, artificial, tenue) | La luz cambia todos los píxeles: entrenado solo con flash, fracasa en el aula | ☐ |
| 4 | **Mínimo 4 fondos** (mesa, suelo, pared lisa, fondo cargado) | Sin variedad de fondos aprende el FONDO, no el objeto (error nº 1 de esta actividad) | ☐ |
| 5 | **Mínimo 3 distancias** (cerca, media, lejos) | Escala: el mismo objeto puede ocupar 50 o 500 píxeles | ☐ |
| 6 | **El objeto NO siempre centrado** ni siempre en la misma esquina | Si siempre está al centro, confundirá posición con identidad | ☐ |
| 7 | **Algunas fotos difíciles a propósito** (parcialmente tapado, junto a objetos parecidos) | El mundo real es sucio: el detector debe ver casos límite | ☐ |
| 8 | **10 fotos APARTE, guardadas en otra carpeta** (`test_manual/`) SIN usarlas para entrenar | Serán vuestro examen: fotos que el modelo jamás ha visto | ☐ |

> ⚠️ **Regla de oro:** si vuestras fotos podrían describirse como "60 fotos del mismo objeto en el mismo sitio con la misma luz", el detector va a fallar. La variedad no es opcional: es el dataset.

## Pregunta 1 · Diagnóstico del dataset (antes de etiquetar)

Antes de subir nada a Roboflow, responded en esta celda de texto:

**(a)** ¿Qué objeto habéis elegido y por qué?

**(b)** Rellenad esta tabla con vuestras fotos reales (contadlas):

| Dimensión | ¿Cuántas variantes tenéis? | Objetivo mínimo |
|---|---|---|
| Ángulos | | 5 |
| Iluminaciones | | 3 |
| Fondos | | 4 |
| Distancias | | 3 |
| Total de fotos | | 60 |

**(c)** ¿Qué dimensión os ha quedado más floja? ¿Qué pasaría en el mundo real si el detector solo hubiera visto esa dimensión tan pobre? (2-3 frases)


# 🏷️ Fase 2 — Etiquetar en Roboflow

## Paso a paso

1. Cread una cuenta gratuita en [roboflow.com](https://roboflow.com) (plan **Public** gratuito: suficiente para clase; se puede usar cuenta de Gmail).
2. **New Project** → tipo *Object Detection* → nombre del proyecto (p. ej. `detector-auriculares`) → *Annotation Group* por grupo.
3. Subid las fotos de entrenamiento (**NO las 10 de `test_manual/`**).
4. Etiquetad: dibujad una caja ajustada alrededor del objeto en cada imagen, con el nombre de la clase (p. ej. `auricular`).
   - Atajo: pulsad `A` para la siguiente imagen sin etiquetar.
   - Si el objeto está parcialmente tapado: caja solo de la parte visible.
5. (**Opcional pero recomendado**) *Generate* → **Augmentation**: añadid rotación (±15°), brillo (±25%) y ruido leve. Roboflow creará variaciones sintéticas: *estamos fabricando la variedad que no fotografiamos*.
6. **Generate → Export Dataset → Format: YOLOv8** → copiad el **código de descarga** que os da (contiene una URL con clave).

> 💡 **Por qué etiquetar es la parte más larga:** en un proyecto real, el 80% del tiempo se va en recopilar y etiquetar datos, no en programar. El modelo será tan bueno como sus etiquetas: *garbage in, garbage out*.

## Sobre el aumento de datos (augmentation)

Cada transformación sintetiza una buena práctica de la checklist:

| Transformación | Equivale a... |
|---|---|
| Rotación ±15° | variedad de ángulos |
| Brillo ±25% | variedad de iluminación |
| Ruido / desenfoque leve | fotos reales imperfectas |
| Recortes aleatorios | objeto no siempre centrado |

⚠️ Usad transformaciones **moderadas**: una rotación de 180° de una taza genera una taza "boca abajo" que quizá nunca veréis en la realidad, y solo añade confusión.


# 🚀 Fase 3 — Entrenar en Colab (con GPU gratis)

Ejecutad las celdas siguientes en orden. **Antes de nada:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.

## Paso 1: instalar Ultralytics

```python
# TU CÓDIGO: instala el paquete ultralytics con pip
# pista: !pip install ultralytics
```

## Paso 2: descargar VUESTRO dataset

Pegad aquí el código de descarga que os dio Roboflow al exportar (botón *"Show download code"* → pestaña Python). Sustituid el ejemplo:

```python
# --- PEGAD AQUÍ VUESTRO CÓDIGO ROBOFLOW (el que os da al exportar) ---
# Tendrá esta pinta:
# from roboflow import Roboflow
# rf = Roboflow(api_key="VUESTRA_CLAVE")
# project = rf.workspace("...").project("detector-auriculares")
# version = project.version(1)
# dataset = version.download("yolov8")
# ---------------------------------------------------------------------
```

## Paso 3: entrenar

```python
# TU CÓDIGO: entrena con yolo
# Usa la CLI de yolo, tarea de DETECCIÓN (no segmentación):
#   data = ruta al data.yaml que Roboflow ha descargado
#   model = yolov8n.pt   (Nano preentrenado: transfer learning)
#   epochs = 40
#   imgsz = 640
# pista: !yolo detect train data=... model=yolov8n.pt epochs=40 imgsz=640
```

## Pregunta 2 · Explicad los parámetros (celda de texto)

**(a)** ¿Qué significa `epochs=40`? ¿Qué hace el modelo en cada época?

**(b)** ¿Qué es `yolov8n.pt` y por qué NO empezamos de cero sino desde un modelo preentrenado? ¿Cómo se llama esa técnica?

**(c)** ¿Para qué sirve la carpeta `valid/` que Roboflow creó dentro del dataset? ¿Qué estaría pasando si la precisión en train fuera altísima y en valid muy baja?

**(d)** Buscad en la salida del entrenamiento la métrica **mAP50** y anotad su valor: ______


In [ ]:
# TU CÓDIGO — Paso 1: instalar Ultralytics
# (recuerda activar la GPU: Entorno de ejecución -> Cambiar tipo -> T4 GPU)


In [ ]:
# TU CÓDIGO — Paso 2: descargar vuestro dataset de Roboflow
# PEGAD AQUÍ el código de descarga que os dio Roboflow al exportar en formato YOLOv8


In [ ]:
# TU CÓDIGO — Paso 3: entrenar el modelo
# pista: !yolo detect train data=<ruta_al_data.yaml> model=yolov8n.pt epochs=40 imgsz=640


# 🧪 Fase 4 — Probar y evaluar

## Paso 4: predicción sobre una imagen

Subid a Colab una de las fotos de vuestra carpeta `test_manual/` (las que el modelo **nunca** ha visto) y ejecutad la predicción:

```python
# TU CÓDIGO
# pista: !yolo detect predict model=runs/detect/train/weights/best.pt source=foto_test.jpg conf=0.25
```

## Paso 5 (el gran momento): detector en directo con la webcam

```python
import cv2
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")
# TU CÓDIGO: capturad de la webcam (cv2.VideoCapture(0)) en un bucle,
# pasad cada frame por model.predict() y mostrad el resultado con
# cv2.imshow("detector", r.plot()). Al pulsar la tecla 'q', salir.
```

## Pregunta 3 · Evaluación honesta (celda de texto)

Probad el detector con las 10 fotos de `test_manual/` y en directo. Después responded:

**(a)** ¿En cuántas de las 10 fotos nuevas detecta correctamente el objeto? ¿Coincide ese porcentaje con el mAP50 que anotasteis en la Pregunta 2? ¿Por qué puede no coincidir?

**(b)** Describid **un fallo real** que hayáis visto (no lo detecta, lo detecta donde no hay, caja mal ajustada...) y relacionadlo con vuestra respuesta de la Pregunta 1(c): ¿tiene que ver con la dimensión floja de vuestro dataset?

**(c)** Probad a mostrar al detector un **objeto parecido pero distinto** (otro tipo de auricular, otra botella...). ¿Qué pasa? ¿Cómo se llama ese fenómeno (pista: el modelo confunde...)?


In [ ]:
# TU CÓDIGO — Paso 4: predicción sobre una foto de test_manual/
# pista: !yolo detect predict model=runs/detect/train/weights/best.pt source=foto_test.jpg conf=0.25


In [ ]:
# TU CÓDIGO — Paso 5: detector en directo con la webcam
import cv2
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")
# Bucle: capturar frame, model.predict(frame), cv2.imshow(r.plot()), salir con 'q'


# 🧠 Pregunta 4 · Reflexión final (celda de texto — 2 puntos)

**(a)** La frase de la actividad era: *"el modelo será tan bueno como sus etiquetas"*. Explicadla con UN ejemplo concreto de vuestro proyecto.

**(b)** Si tuvierais que detectar vuestro objeto en **las 200 tiendas de una cadena nacional** con cámaras en directo (24/7, miles de imágenes por segundo), ¿qué partes de vuestro pipeline habría que rediseñar? Nombrad al menos dos y justificadlo (pista: pensad en volumen, en dónde se almacenan las imágenes y en cuánto cuesta procesarlas... ¿os suena la diferencia entre un ordenador y un **sistema Big Data**?).

**(c)** Una clínica quiere usar un sistema así para ayudar a detectar estructuras en ecografías (como el artículo de Linux Magazine). Escribid **dos** precauciones legales o éticas que deberían tener antes de ponerlo en marcha (pista: datos de salud, errores del modelo, quién decide...).

---

## 📊 Registro de resultados del grupo

| Métrica | Valor |
|---|---|
| Nº de fotos de entrenamiento (con aumento) | |
| Nº de fotos de test manual | |
| Épocas entrenadas | |
| mAP50 final | |
| Aciertos en las 10 fotos de test | /10 |

> ✍️ **Entrega:** guardad este cuaderno como `detector_objeto_NOMBREGRUPO.ipynb` con todas las celdas ejecutadas y las preguntas de texto respondidas.
